# Factual Consistency Checking (Q&A Method)

This notebook implements a factual consistency checker using a Question Generation (QG) and Question Answering (QA) pipeline, as requested.

### Methodology
1.  **Source of Truth**: The original essay (`datasets/essays/...` or `AI_essay` column).
2.  **Candidate Text**: The humanized essay (`datasets/humanized_essays/...` or `humanized_essay` column).
3.  **Process**:
    *   **Step 1 (Generate)**: Generate factual questions (and ground truth answers) based *only* on the **Original Essay**.
    *   **Step 2 (Answer)**: Answer these generated questions using the **Humanized Essay** as the context.
    *   **Step 3 (Verify)**: Compare the "Humanized Answer" against the "Ground Truth Answer" to check for factual consistency (Correctness).

*Note: This approach differs from the standard [FActScore](https://github.com/shmsw25/FActScore) library, which breaks text into atomic facts and verifies them against Wikipedia. Instead, this Q&A method (similar to QAGS/QuestEval) treats the Original Essay as the knowledge source.*

In [1]:
import pandas as pd
import os
import json
import time
from tqdm import tqdm

# Load the dataset
# Assuming datasets.csv is in the parent directory
dataset_path = '../datasets.csv'

if os.path.exists(dataset_path):
    df = pd.read_csv(dataset_path)
    print(f"Dataset loaded successfully. Shape: {df.shape}")
    display(df.head(3))
else:
    print(f"Error: {dataset_path} not found. Please check the path.")

Dataset loaded successfully. Shape: (180, 6)


,topic,length,AI_model,humanizer,AI_essay,humanized_essay
0,1,1,gemini2.5pro,AIHumanizer,The integration of social media into the lives...,The way social media has woven itself into the...
1,1,1,gemini2.5pro,Grammarly,The integration of social media into the lives...,Social media has become a big part of young pe...
2,1,1,gemini2.5pro,HumanizeAI,The integration of social media into the lives...,The penetration of social media into the lives...


In [2]:
# drop error row (case when AI = 'gemini2.5pro', humanizer = 'writehuman.ai', topic = 5, length = 2)
df = df[~((df['AI_model'] == 'gemini2.5pro') & (df['humanizer'] == 'writehuman.ai') & (df['topic'] == 5) & (df['length'] == 2))].reset_index(drop=True)
print(f"Dataset shape after dropping error row: {df.shape}")
display(df.tail(4))

Dataset shape after dropping error row: (179, 6)


,topic,length,AI_model,humanizer,AI_essay,humanized_essay
175,5,3,gpt4.0,HumanizeAI,Childhood is often remembered through the lens...,Childhood is often remembered as a realm of si...
176,5,3,gpt4.0,Quillbot,Childhood is often remembered through the lens...,Childhood is frequently viewed through the pri...
177,5,3,gpt4.0,UndetectableAI,Childhood is often remembered through the lens...,People tend to recall their childhood through ...
178,5,3,gpt4.0,writehuman.ai,Childhood is often remembered through the lens...,Childhood is remembered as a time of simplicit...


In [3]:
from dotenv import load_dotenv
load_dotenv()
API_KEY = os.getenv("OPENAI_API_KEY")
API_ENDPOINT = os.getenv("OPENAI_API_ENDPOINT")
DEPLOYMENT_NAME = os.getenv("GPT5MINI_DEPLOYMENT_NAME")
API_VERSION = os.getenv("GPT5MINI_DEPLOYMENT_VERSION")

In [4]:
API_VERSION

'2024-12-01-preview'

In [5]:
# --- LLM Client Setup ---
import os
import time
from openai import AzureOpenAI

client = AzureOpenAI(
    api_version=API_VERSION,
    azure_endpoint=API_ENDPOINT,
    api_key=API_KEY,
)

def call_llm(prompt, model="gpt-5-mini"): 
    """
    Calls Gemini API using the google-genai library.
    Added rate limiting to respect 10 calls/minute.
    """
    # Rate Limiting
    time.sleep(0.5)
    
    try:
        if model == "gpt-5-mini":
            deployment = DEPLOYMENT_NAME  # Use deployment name from environment variable
            response = client.chat.completions.create(
                model=deployment,
                messages=[{"role": "user", "content": prompt}],
            )
            return response.choices[0].message.content
        else:
            raise ValueError("Unsupported model specified.")
    except Exception as e:
        print(f"Error calling LLM: {e}")
        return "{}"

In [6]:
# --- Core Logic: QG -> QA -> Evaluation ---
def generate_questions_from_original(original_text, num_questions=3):
    """
    Step 1: Generate questions and ground truth answers from the Original Essay.
    """
    prompt = f"""
    You are a factual consistency evaluator. 
    Read the following text (Original Essay) and generate {num_questions} specific, factual questions that can be answered based on the text.
    For each question, provide the correct answer as found in the text.
    
    Output format must be valid JSON:
    [
        {{"question": "Question 1?", "answer": "Answer 1"}},
        {{"question": "Question 2?", "answer": "Answer 2"}}
    ]

    Original Essay:
    {original_text}
    """
    response = call_llm(prompt)
    
    # Parse JSON (Add error handling in production)
    try:
        # cleanup markdown code blocks if any
        clean_response = response.replace('```json', '').replace('```', '').strip()
        qa_pairs = json.loads(clean_response)
    except:
        qa_pairs = [] # Fallback
        
    return qa_pairs

def answer_with_humanized(humanized_text, question):
    """
    Step 2: Answer the question using the Humanized Essay as context.
    """
    prompt = f"""
    Answer the question based ONLY on the provided Context. 
    If the information is not present in the context, say "Information not found".

    Context (Humanized Essay):
    {humanized_text}

    Question:
    {question}

    Answer:
    """
    return call_llm(prompt)

def evaluate_answer_match(ground_truth, candidate_answer, question):
    """
    Step 3: Check if the candidate answer matches the ground truth.
    """
    prompt = f"""
    Question: {question}
    
    Ground Truth Answer (from Original): {ground_truth}
    Candidate Answer (from Humanized): {candidate_answer}

    Task: Determine if the Candidate Answer is factually consistent with the Ground Truth Answer.
    Does the Humanized text (Candidate) preserve the key fact?
    
    Respond with JSON:
    {{"is_consistent": true/false, "reason": "Explanation"}}
    """
    response = call_llm(prompt)
    try:
        clean_response = response.replace('```json', '').replace('```', '').strip()
        result = json.loads(clean_response)
        return result
    except:
        return {"is_consistent": False, "reason": "Error parsing LLM response"}

def check_factual_score_for_row(qa_pairs, humanized_text):
    # 1. Generate Qs
    if not qa_pairs:
        return 0.0, []

    results = []
    correct_count = 0
    
    # 2. Key Loop
    for item in qa_pairs:
        q = item['question']
        gt_a = item['answer']
        
        # Answer with Humanized
        cand_a = answer_with_humanized(humanized_text, q)
        
        # Verify
        eval_res = evaluate_answer_match(gt_a, cand_a, q)
        
        entry = {
            "question": q,
            "ground_truth": gt_a,
            "humanized_answer": cand_a,
            "is_consistent": eval_res.get("is_consistent", False),
            "reason": eval_res.get("reason", "")
        }
        results.append(entry)
        
        if entry["is_consistent"]:
            correct_count += 1
            
    score = correct_count / len(qa_pairs) if qa_pairs else 0
    return score, results


In [7]:
# --- Run Evaluation on a Sample ---

# Group by 'topic' and 'AI_model' to avoid regenerating questions for the same original essay repeatedly
# For demonstration, we'll just process the first 2 rows of the DataFrame directly.

sample_df = df.head(2).copy() # Process first 2 rows for testing
output_results = []
processed_AI_essays = dict()

print("Starting Evaluation Loop...")

for index, row in tqdm(sample_df.iterrows(), total=len(sample_df)):
    original = row['AI_essay']
    humanized = row['humanized_essay']
    if original in processed_AI_essays:
        qa_pairs = processed_AI_essays[original]
    else:
        qa_pairs = generate_questions_from_original(original, num_questions=3)
        print(f"created new questions & answers pair for {row['length']} - {row['topic']} - {row['AI_model']}")
        processed_AI_essays[original] = qa_pairs
    
    score, details = check_factual_score_for_row(qa_pairs, humanized)
    
    # Store result
    output_results.append({
        "topic": row['topic'],
        "humanizer": row['humanizer'],
        "factual_consistency_score": score,
        "details": details
    })

# Convert to DataFrame
results_df = pd.DataFrame(output_results)
print("\nEvaluation Results:")
display(results_df)

Starting Evaluation Loop...


  0%|          | 0/2 [00:00<?, ?it/s]

created new questions & answers pair for 1 - 1 - gemini2.5pro


100%|██████████| 2/2 [00:55<00:00, 27.91s/it]


Evaluation Results:


,topic,humanizer,factual_consistency_score,details
0,1,AIHumanizer,1.0,"[{'question': 'According to the essay, what is..."
1,1,Grammarly,1.0,"[{'question': 'According to the essay, what is..."


In [8]:
results_df.loc[0, 'details']

[{'question': 'According to the essay, what is the primary effect of social media on youth?',
  'ground_truth': 'profoundly detrimental',
  'humanized_answer': 'It has a largely harmful effect—undermining mental health and real social development, increasing anxiety, depression and body‑image issues, and weakening social skills and self‑esteem.',
  'is_consistent': True,
  'reason': "Yes — the candidate states social media has a largely harmful effect (undermining mental health and social development, increasing anxiety, depression, body‑image issues, and weakening social skills and self‑esteem), which aligns with the ground truth 'profoundly detrimental.'"},
 {'question': 'What does the essay identify as the core of the problem with social media?',
  'ground_truth': 'the relentless pressure of social comparison and the metrics of validation that govern the digital space',
  'humanized_answer': 'The essay says the core problem is the constant pressure to compare oneself to others, driv

In [ ]:
# real run on next 6 rows with caching of generated questions
df_temp = df.iloc[12:18].copy() # Process next 6 rows for testing
processed_AI_essays = dict()
output_results = []
for index, row in tqdm(df_temp.iterrows(), total=len(df_temp)):
    original = row['AI_essay']
    humanized = row['humanized_essay']
    if original in processed_AI_essays:
        qa_pairs = processed_AI_essays[original]
    else:
        qa_pairs = generate_questions_from_original(original, num_questions=20)
        print(f"created new questions & answers pair for {row['length']} - {row['topic']} - {row['AI_model']}")
        processed_AI_essays[original] = qa_pairs
    
    score, details = check_factual_score_for_row(qa_pairs, humanized)
    
    temp_df = pd.DataFrame([{
        "topic": row['topic'],
        "length": row['length'],
        "AI_model": row['AI_model'],
        "humanizer": row['humanizer'],
        "factual_consistency_score": score,
        "details": details
    }])
    temp_df.to_csv('qna_results.csv', mode='a',index=False, header=False)
    print(f"save result for {row['length']} - {row['topic']} - {row['AI_model']} - {row['humanizer']}")
    
    output_results.append({
        "topic": row['topic'],
        "length": row['length'],
        "AI_model": row['AI_model'],
        "humanizer": row['humanizer'],
        "factual_consistency_score": score,
        "details": details
    })
results_df = pd.DataFrame(output_results)
print("\nEvaluation Results:")
display(results_df)

  0%|          | 0/6 [00:00<?, ?it/s]

created new questions & answers pair for 2 - 1 - gemini2.5pro


 17%|█▋        | 1/6 [03:06<15:32, 186.46s/it]

save result for 2 - 1 - gemini2.5pro - AIHumanizer


 33%|███▎      | 2/6 [05:38<11:05, 166.41s/it]

...

 83%|████████▎ | 5/6 [13:40<02:40, 160.08s/it]

save result for 2 - 1 - gemini2.5pro - UndetectableAI


100%|██████████| 6/6 [16:16<00:00, 162.78s/it]

save result for 2 - 1 - gemini2.5pro - writehuman.ai

Evaluation Results:


,topic,length,AI_model,humanizer,factual_consistency_score,details
0,1,1,gemini2.5pro,AIHumanizer,0.95,[{'question': 'How does the essay describe the...
1,1,1,gemini2.5pro,Grammarly,0.80,[{'question': 'How does the essay describe the...
2,1,1,gemini2.5pro,HumanizeAI,0.85,[{'question': 'How does the essay describe the...
3,1,1,gemini2.5pro,Quillbot,0.90,[{'question': 'How does the essay describe the...
4,1,1,gemini2.5pro,UndetectableAI,0.80,[{'question': 'How does the essay describe the...
5,1,1,gemini2.5pro,writehuman.ai,0.65,[{'question': 'How does the essay describe the...
6,1,1,gpt4.0,AIHumanizer,0.95,[{'question': 'What has social media transform...
7,1,1,gpt4.0,Grammarly,1.00,[{'question': 'What has social media transform...
8,1,1,gpt4.0,HumanizeAI,0.85,[{'question': 'What has social media transform...
9,1,1,gpt4.0,Quillbot,0.90,[{'question': 'What has social media transform...


In [16]:
# read qna_results.csv to see all results
results_df = pd.read_csv('qna_results.csv')
display(results_df)

,topic,length,AI_model,humanizer,factual_consistency_score,details
0,1,1,gemini2.5pro,AIHumanizer,0.95,[{'question': 'How does the essay describe the...
1,1,1,gemini2.5pro,Grammarly,0.80,[{'question': 'How does the essay describe the...
2,1,1,gemini2.5pro,HumanizeAI,0.85,[{'question': 'How does the essay describe the...
3,1,1,gemini2.5pro,Quillbot,0.90,[{'question': 'How does the essay describe the...
4,1,1,gemini2.5pro,UndetectableAI,0.80,[{'question': 'How does the essay describe the...
5,1,1,gemini2.5pro,writehuman.ai,0.65,[{'question': 'How does the essay describe the...
6,1,1,gpt4.0,AIHumanizer,0.95,[{'question': 'What has social media transform...
7,1,1,gpt4.0,Grammarly,1.00,[{'question': 'What has social media transform...
8,1,1,gpt4.0,HumanizeAI,0.85,[{'question': 'What has social media transform...
9,1,1,gpt4.0,Quillbot,0.90,[{'question': 'What has social media transform...
